<a href="https://colab.research.google.com/github/voraciousnerd/iqm-quantum-school-labs/blob/main/QS26_Day3_Lab_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **From abstract code to physical pulses**

# Setup

Install the appropriate packages for Python by running the following cell and restart the session as prompted (without re-running this cell).

In [ ]:
!pip list --format=freeze | grep -iE '^(google-colab|ipykernel|jupyter-core)=' > /content/colab-pins.txt
!pip install -c /content/colab-pins.txt "iqm-pulla[pin-iqm]==14.0.0" ipython==8.21.0 "qrisp[iqm]==0.9.6"

# Part 2: Compile quantum circuits into pulses
In this first exercise, you will get an idea of unitary gate implementation through direct visualization of pulses.

In this notebook, you will ...
* Prepare a GHZ state and compile the corresponding quantum circuit into a *playlist* of microwave pulses.

By the end of this notebook, you will understand the correspondence between unitary gates and their physical implementation for superconducting qubits.

In order to get started, make sure you have the appropriate packages installed:

## 1. Preparation

### 1.1 Connecting to the QPU station control
In the following, we will use the PulLa (Pulse Level Access) package. You can find the documentation [here](https://docs.iqm.tech/4.6/iqm-pulla/index.html
).

As a first step, we need to create a **PulLa object**. Conceptually, this is an IQM quantum computer client for connecting to the IQM server and constructing a circuit-to-pulse compiler. A compiler object defines the specific circuit-to-pulse compilation logic; it contains information about the quantum computer, like chip topology, the set of available native operations and other details that can be found [here](https://docs.iqm.tech/4.6/iqm-pulla/Configuration%20and%20Usage.html).

Make sure you have the correct url and token and run the cell below. By default,  the `get_standard_compiler()` function fetches the default calibration set from the server:

In [ ]:
from iqm.pulla.pulla import Pulla

p = Pulla("https://resonance.iqm.tech", quantum_computer="garnet", token=input("Enter your Resonance API token: "))
compiler = p.get_standard_compiler()

## 2. GHZ state

You have learnt in the past labs and exercises what a GHZ state is and how to prepare one on a quantum computer. Here, we are not interested in its entanglement properties, but we will use it as a starting point to look into physical pulses. Let's construct a GHZ for 3 qubits with Qrisp (documentation [here](https://qrisp.eu/index.html)):

In [ ]:
from qrisp import QuantumVariable, h, cx

qv = QuantumVariable()
# TODO: construct a GHZ for 3 qubits

qc_0 = qv.qs

print(qc_0)

Before compiling the circuit, we need to transpile it to take into account the native gates and connectivity of IQM quantum computers (this may take a minute to run):

In [ ]:
from iqm.qrisp_iqm import transpile_to_iqm, IQMBackend

backend = IQMBackend("https://resonance.iqm.tech", device_instance="garnet", token=input("Enter your Resonance API token: "))

coupling_map = backend.connectivity
transpiled_qc = transpile_to_iqm(qc_0, coupling_map)
print(transpiled_qc)

## 3. Compilation

We can now extract the list of *instructions* from the circuit defined above and `compile` it to obtain a *playlist* of pulses!

To do that, we use `qrisp_to_iqm_converter` to obtain the circuit in Pulla format.

In [ ]:
from iqm.iqm_client import IQMClient
from iqm.qrisp_iqm import qrisp_to_iqm_converter

client = IQMClient("https://resonance.iqm.tech", quantum_computer="garnet", token=input("Enter your Resonance API token: "))
dqa = client.get_dynamic_quantum_architecture()
iqm_circuit = qrisp_to_iqm_converter(transpiled_qc, dqa)
job_definition, context = compiler.compile([iqm_circuit])

Finally, we are ready to visualize the pulses that prepare a GHZ state of 3 qubits with `inspect_playlist`. Let's explore the details!

**Q: What is the segment corresponding to the single-qubit gates in the table below?**

In [ ]:
from iqm.pulse.playlist.visualisation.base import inspect_playlist
from IPython.display import HTML, display

html_content = inspect_playlist(job_definition.sweep_definition.playlist)
display(HTML(html_content))